# Submit 01 to RSNA Knee Abnormality Detection

1. Build the cache representation 
2. Run the ConvNeXt-Tiny 2.5D Model
3. Ensemble over every attached set of fold weights 
4. Write `submission.csv`.

## Data Preprocessing

Turn DICOMS into cahced representation

In [ ]:
# Measure Time
import time

NB_T0 = time.time()
print("kernel start")

## Config

In [ ]:
CFG = {
    "COMP_DIR": "/kaggle/input/rsna-knee-abnormality-detection",
    "SPLIT": "test",
    "OUT_DIR": "/tmp/cache",

    "SLOTS": ["SAG_FS", "COR_FS", "AX_FS", "SAG_NOFS", "COR_T1", "SAG_T1"],
    "N_SLICES": 24,
    "BAND": (0.10, 0.90), # fraction of the ordered stack to sample from
    "IMG": 320,
    "CROP_MM": 140.0, # physical square centre crop before resize
    "CLIP_LO": 0.5, # per-series percentile clip on the cropped stack -> uint8
    "CLIP_HI": 99.5,
    "MIN_SLICES": 6,
    "LAT_DEAD_MM": 20.0,
    "COMPRESS": True,

    "N_SHARDS": 1,
    "SHARD_INDEX": 0,

    "N_WORKERS": 0, 
    "TIME_BUDGET_H": 4.0,
    "SEED": 42,

    "DEBUG": False,
    "DEBUG_N": 20,
}
N_SLOTS = len(CFG["SLOTS"])

## Imports and setup

In [ ]:
import os
import re
import sys
import time
import zlib
import random
import traceback
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm

random.seed(CFG["SEED"])
np.random.seed(CFG["SEED"])
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")

try:
    import cv2
    cv2.setNumThreads(0)
    def resize(a, size):
        return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)
except ImportError:
    from PIL import Image
    def resize(a, size):
        return np.asarray(Image.fromarray(a).resize((size, size), Image.BILINEAR))

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pydicom", pydicom.__version__)
print("cpus:", os.cpu_count())

In [ ]:
def find_root(comp_dir):
    """Locate the mount. Should be a dir holding {split}.csv and {split}_series/."""
    split = CFG["SPLIT"]
    candidates = [
        Path(comp_dir),
        Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("data"),
        Path("."),
    ]
    def ok(c):
        return (c / f"{split}.csv").is_file() and (c / f"{split}_series").is_dir()
    for c in candidates:
        if ok(c):
            return c
    base = Path("/kaggle/input")
    if base.is_dir():
        for d1 in sorted(p for p in base.iterdir() if p.is_dir()):
            for c in [d1] + sorted(p for p in d1.iterdir() if p.is_dir()):
                if ok(c):
                    return c
    raise FileNotFoundError(f"competition not found.")

ROOT = find_root(CFG["COMP_DIR"])
SPLIT = CFG["SPLIT"]
SERIES_DIR = ROOT / f"{SPLIT}_series"
OUT_DIR = Path(CFG["OUT_DIR"])
OUT_DIR.mkdir(parents=True, exist_ok=True)

series_df = pd.read_csv(ROOT / f"{SPLIT}_series.csv")
STUDY_ROWS = {s: g.to_dict("records") for s, g in series_df.groupby("StudyInstanceUID")}
print("root:", ROOT)
print("series rows:", len(series_df), "| studies:", len(STUDY_ROWS))

## Parse Header 

Read things like contrast, fat suppression, geometry, laterality

In [ ]:
HDR_TAGS = [
    "SeriesDescription", "ProtocolName", "SequenceName", "ScanningSequence", "ScanOptions",
    "RepetitionTime", "EchoTime", "InversionTime", "Laterality", "ImageLaterality",
    "PixelSpacing", "Rows", "Columns", "ImageOrientationPatient", "ImagePositionPatient",
    "PhotometricInterpretation", "SeriesInstanceUID",
]
_RX_LOC = re.compile(r"loc|scout|survey|plan|calib|smartbrain|3.?plane", re.I)
_RX_T1 = re.compile(r"\bt1\b|t1w|t1_|_t1|t1 ", re.I)
_RX_T2 = re.compile(r"\bt2\b|t2w|t2_|_t2|t2 ", re.I)
_RX_PD = re.compile(r"\bpd\b|\bdp\b|proton|dens|pdw", re.I)
_RX_STIR = re.compile(r"stir|tirm", re.I)
_RX_FS = re.compile(r"\bfs\b|fat.?sat|spir|spair|fatsup|\bsat\b|fs_|_fs|fs ", re.I)


def _f(x, default=None):
    try:
        return float(x)
    except (TypeError, ValueError):
        return default


def read_header(series_path):
    """Parse one series' header into a plain dict. Never raises."""
    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))
    h = {"n_files": len(files), "hdr_ok": 0}
    if not files:
        return h
    try:
        ds = pydicom.dcmread(str(files[len(files) // 2]), stop_before_pixels=True,
                             force=True, specific_tags=HDR_TAGS)
    except Exception:
        return h
    h["hdr_ok"] = 1
    h["desc"] = " ".join(str(getattr(ds, t, "") or "") for t in
                         ("SeriesDescription", "ProtocolName", "SequenceName"))
    h["scan_seq"] = str(getattr(ds, "ScanningSequence", "") or "")
    h["scan_opts"] = str(getattr(ds, "ScanOptions", "") or "")
    h["tr"] = _f(getattr(ds, "RepetitionTime", None))
    h["te"] = _f(getattr(ds, "EchoTime", None))
    h["ti"] = _f(getattr(ds, "InversionTime", None))
    lat = str(getattr(ds, "ImageLaterality", "") or getattr(ds, "Laterality", "") or "").strip().upper()
    h["lat_tag"] = lat if lat in ("L", "R") else ""
    ps = getattr(ds, "PixelSpacing", None)
    h["ps_row"] = _f(ps[0]) if ps is not None and len(ps) >= 2 else None
    h["ps_col"] = _f(ps[1]) if ps is not None and len(ps) >= 2 else None
    h["rows"] = int(getattr(ds, "Rows", 0) or 0)
    h["cols"] = int(getattr(ds, "Columns", 0) or 0)
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    h["iop"] = [float(x) for x in iop[:6]] if iop is not None and len(iop) >= 6 else None
    h["ipp"] = [float(x) for x in ipp[:3]] if ipp is not None and len(ipp) >= 3 else None
    h["mono1"] = str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1"
    h["centre_x"] = image_centre_x(h)
    return h


def image_centre_x(h):
    """Patient-x (LPS: +x = patient's left) of the image centre, in mm. None if unknown."""
    if not h.get("iop") or not h.get("ipp") or not h.get("ps_row") or not h.get("rows"):
        return None
    r = np.asarray(h["iop"][:3])   # direction of increasing column index
    c = np.asarray(h["iop"][3:])   # direction of increasing row index
    centre = (np.asarray(h["ipp"]) + r * (h["cols"] - 1) / 2 * h["ps_col"]
              + c * (h["rows"] - 1) / 2 * h["ps_row"])
    return float(centre[0])


def classify_contrast(h, csv_fs):
    """-> (contrast in {T1, PD, T2, STIR, GRE, UNK}, fs: bool)."""
    desc = h.get("desc", "") or ""
    opts = {t.strip().upper() for t in re.split(r"[\\/,;]", h.get("scan_opts") or "")}
    fs = bool(csv_fs) or bool(_RX_FS.search(desc)) or bool(opts & {"FS", "FAT_SAT", "FATSAT", "SPIR", "SPAIR"})
    if _RX_STIR.search(desc) or (h.get("ti") and 80 <= h["ti"] <= 250):
        return "STIR", True
    if _RX_T1.search(desc):
        return "T1", fs
    if _RX_T2.search(desc):
        return "T2", fs
    if _RX_PD.search(desc):
        return "PD", fs
    tr, te = h.get("tr"), h.get("te")
    if tr is not None and te is not None:
        if tr < 800:
            return "T1", fs
        if te > 60:
            return "T2", fs
        return "PD", fs
    if "GR" in (h.get("scan_seq") or ""):
        return "GRE", fs
    return "UNK", fs


def is_localizer(h):
    return h.get("n_files", 0) < CFG["MIN_SLICES"] or bool(_RX_LOC.search(h.get("desc", "") or ""))

## Slot selection

Each slot has a (plane, fs, contrast) requirement. Candidates are ranked by slice count. 
A series fills at most one slot. Slots are filled in the order listed.

In [ ]:
SLOT_SPEC = {
    "SAG_FS":   ("Sagittal", True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "COR_FS":   ("Coronal",  True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "AX_FS":    ("Axial",    True,  {"PD", "T2", "STIR", "UNK", "GRE"}),
    "SAG_NOFS": ("Sagittal", False, {"PD", "T2", "GRE", "UNK"}),
    "COR_T1":   ("Coronal",  None,  {"T1"}),
    "SAG_T1":   ("Sagittal", None,  {"T1"}),
}


FALLBACK_ORDER = ["T2", "PD", "GRE", "UNK"] 

def pick_slots(meta):
    """
    Pick slots.
    
    Args:
        meta - list of dicts with uid, plane, fs, contrast, n_files, localizer.

    Return:
    (uids, fallback) — two lists of len N_SLOTS in CFG["SLOTS"] order; uid None = empty.

    Pass 1 fills every slot from series that match its spec exactly. 
    Pass 2 fills any FS slot still empty with a leftover same-plane non-FS T2, then PD.
    """

    used, chosen, fallback = set(), [], [False] * N_SLOTS

    for name in CFG["SLOTS"]:

        plane, want_fs, contrasts = SLOT_SPEC[name]
        cands = [m for m in meta
                 if m["uid"] not in used and not m["localizer"] and m["plane"] == plane
                 and m["contrast"] in contrasts
                 and (want_fs is None or bool(m["fs"]) == want_fs)]
        cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
        pick = cands[0]["uid"] if cands else None

        if pick is not None:
            used.add(pick)

        chosen.append(pick)

    for i, name in enumerate(CFG["SLOTS"]):
        plane, want_fs, _ = SLOT_SPEC[name]

        if chosen[i] is not None or want_fs is not True:
            continue

        for con in FALLBACK_ORDER:
            cands = [m for m in meta if m["uid"] not in used and not m["localizer"]
                     and m["plane"] == plane and m["contrast"] == con]

            if cands:
                cands.sort(key=lambda m: (-m["n_files"], m["uid"]))
                chosen[i] = cands[0]["uid"]; used.add(chosen[i]); fallback[i] = True
                break

    return chosen, fallback


def study_laterality(meta):
    """-> (side in {L, R, U}, source). Tag majority first, then image-centre x."""
    
    tags = [m["lat_tag"] for m in meta if m.get("lat_tag")]
    if tags:
        side = max(("L", "R"), key=tags.count)
        return side, "tag"
    xs = [m["centre_x"] for m in meta if m.get("centre_x") is not None]
    if xs:
        x = float(np.median(xs))
        if abs(x) >= CFG["LAT_DEAD_MM"]:
            return ("L" if x > 0 else "R"), "geom"
        return "U", f"geom_dead:{x:.0f}"
    return "U", "none"

## Pixel loading, canonical orientation, physical crop

Slices are ordered by projecting `ImagePositionPatient` onto the slice normal. 
The volume is then flipped so that every axis increases toward +x (left), +y (posterior) or +z
(superior), whichever the axis is aligned with. 
For a **right** knee the axis aligned with x is mirrored.
The crop is a fixed 140 mm square around the image centre.

In [ ]:
ORDER_TAGS = ["ImagePositionPatient", "ImageOrientationPatient", "InstanceNumber"]


def order_files(series_path):
    """(sorted .dcm paths in through-plane order, unit normal or None)."""

    files = sorted(p for p in Path(series_path).iterdir() if p.name.lower().endswith(".dcm"))

    if not files:
        return [], None

    keys, n_geom, normal = [], 0, None

    for i, f in enumerate(files):
        k = None
        try:
            ds = pydicom.dcmread(str(f), stop_before_pixels=True, force=True, specific_tags=ORDER_TAGS)
            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)

            if ipp is not None and iop is not None and len(ipp) >= 3 and len(iop) >= 6:
                p = np.asarray([float(x) for x in ipp[:3]])
                o = np.asarray([float(x) for x in iop[:6]])
                n = np.cross(o[:3], o[3:])
                k = float(np.dot(p, n))

                if np.isfinite(k):
                    n_geom += 1
                    normal = n
                else:
                    k = None

            if k is None:
                inst = getattr(ds, "InstanceNumber", None)
                if inst is not None:
                    k = float(inst)

        except Exception:
            k = None
        keys.append(k if k is not None else float(i))

    if 0 < n_geom < 0.8 * len(files):
        return files, None

    return [f for _, f in sorted(zip(keys, files), key=lambda t: t[0])], normal


def sample_indices(n, k, band):

    if n <= 0:
        return np.zeros(k, dtype=int)
    lo, hi = band

    return np.linspace(lo * (n - 1), hi * (n - 1), k).round().astype(int)


def read_pixels(path):
    try:
        ds = pydicom.dcmread(str(path), force=True)
        a = ds.pixel_array.astype(np.float32)
        if a.ndim == 3:
            a = a[0] if a.shape[0] < a.shape[-1] else a[..., 0]
        a = a * float(getattr(ds, "RescaleSlope", 1.0) or 1.0) + float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
        if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
            a = -a
        return a
    except Exception:
        return None


def crop_mm(a, ps_row, ps_col, size_mm):
    """Centre crop to size_mm x size_mm (zero-pad with the slice min if the FOV is smaller)."""
    if not ps_row or not ps_col:
        return a
    hr, hc = int(round(size_mm / ps_row)), int(round(size_mm / ps_col))
    H, W = a.shape
    r0, c0 = (H - hr) // 2, (W - hc) // 2
    out = np.full((hr, hc), float(a.min()), dtype=np.float32)
    sr0, sc0 = max(r0, 0), max(c0, 0)
    sr1, sc1 = min(r0 + hr, H), min(c0 + hc, W)
    out[sr0 - r0: sr1 - r0, sc0 - c0: sc1 - c0] = a[sr0:sr1, sc0:sc1]
    return out


CANON_SIGN = {0: 1.0, 1: 1.0, 2: -1.0}   # LPS: +x left, +y posterior, -z inferior


def canonical_flips(iop, normal, side):
    """Which axes of (slice, row, col) to flip so the volume is in canonical orientation
    (standard radiological display: columns run right->left or anterior->posterior, rows
    run superior->inferior or anterior->posterior, slices ascend toward +x/+y/-z), then
    mirrored to the left-knee convention when side == 'R'."""
    if iop is None:
        return []
    col_dir = np.asarray(iop[:3])   # increasing column index
    row_dir = np.asarray(iop[3:])   # increasing row index
    dirs = {0: normal, 1: row_dir, 2: col_dir}
    flips = []
    for ax, d in dirs.items():
        if d is None:
            continue
        dom = int(np.argmax(np.abs(d)))
        if d[dom] * CANON_SIGN[dom] < 0:
            flips.append(ax)
        if side == "R" and dom == 0:        # x-aligned axis: mirror right knees
            flips = [f for f in flips if f != ax] if ax in flips else flips + [ax]
    return flips


def load_series(series_path, h, side, n_slices, img, band, crop, clip_lo, clip_hi):
    """(n_slices, img, img) uint8 or None, plus a note."""

    files, normal = order_files(series_path)

    if not files:
        return None, "no_dcm"

    idx = sample_indices(len(files), n_slices, band)
    planes, bad = [], 0

    for i in idx:
        a = read_pixels(files[int(i)])
        if a is None:
            bad += 1
            planes.append(None)
            continue
        planes.append(resize(crop_mm(a, h.get("ps_row"), h.get("ps_col"), crop), img).astype(np.float32))

    good = [p for p in planes if p is not None]

    if not good:
        return None, "all_slices_failed"

    last = None

    for j, p in enumerate(planes):
        if p is None:
            planes[j] = last if last is not None else good[0]
        else:
            last = p

    vol = np.stack(planes, axis=0)

    for ax in canonical_flips(h.get("iop"), normal, side):
        vol = np.flip(vol, axis=ax)

    lo, hi = np.percentile(vol, (clip_lo, clip_hi))
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)

    return np.ascontiguousarray((vol * 255.0).round().astype(np.uint8)), (f"partial:{bad}" if bad else "ok")

## Per-study worker

In [ ]:
W = dict(CFG)
W["SERIES_DIR"] = str(SERIES_DIR)
W["OUT_DIR"] = str(OUT_DIR)


def out_path(study):
    return Path(W["OUT_DIR"]) / f"{study}.npz"


def process_study(study):
    """Build one study's cache file.
    
    Returns (status dict, series-meta rows)."""

    dst = out_path(study)
    rec = {"StudyInstanceUID": study, "status": "ok", "slots": "", "side": "", "side_src": "", "fallback": "", "note": ""}
    meta_rows = []

    try:
        if dst.is_file() and dst.stat().st_size > 0:
            rec["status"] = "skip"
            return rec, meta_rows

        meta = []

        for r in STUDY_ROWS.get(study, []):

            uid = str(r["SeriesInstanceUID"])
            path = Path(W["SERIES_DIR"]) / str(study) / uid
            h = read_header(path) if path.is_dir() else {"n_files": 0, "hdr_ok": 0}
            contrast, fs = classify_contrast(h, r.get("Fluid_Sensitive", 0))
            m = {"uid": uid, "plane": str(r.get("Anatomical_Plane", "")), "fs": fs, "contrast": contrast,
                 "n_files": h.get("n_files", 0), "localizer": is_localizer(h),
                 "lat_tag": h.get("lat_tag", ""), "centre_x": h.get("centre_x"), "h": h}
            meta.append(m)
            meta_rows.append({"StudyInstanceUID": study, "SeriesInstanceUID": uid, "plane": m["plane"],
                              "csv_fs": int(r.get("Fluid_Sensitive", 0) or 0), "fs": int(fs),
                              "contrast": contrast, "n_files": m["n_files"], "localizer": int(m["localizer"]),
                              "tr": h.get("tr"), "te": h.get("te"), "lat_tag": m["lat_tag"],
                              "centre_x": m["centre_x"], "ps_row": h.get("ps_row"),
                              "desc": (h.get("desc") or "")[:80]})

        side, side_src = study_laterality(meta)
        picks, fallback = pick_slots(meta)
        by_uid = {m["uid"]: m for m in meta}

        arr = np.zeros((N_SLOTS, W["N_SLICES"], W["IMG"], W["IMG"]), dtype=np.uint8)
        present = np.zeros(N_SLOTS, dtype=np.uint8)
        notes = []

        for s, uid in enumerate(picks):

            if uid is None:
                continue

            path = Path(W["SERIES_DIR"]) / str(study) / uid

            try:
                vol, note = load_series(path, by_uid[uid]["h"], side, W["N_SLICES"], W["IMG"],
                                        W["BAND"], W["CROP_MM"], W["CLIP_LO"], W["CLIP_HI"])

            except Exception as e:
                vol, note = None, f"err:{type(e).__name__}"

            if vol is None:
                notes.append(f"{s}:{note}")
                continue

            arr[s] = vol
            present[s] = 1

            if note != "ok":
                notes.append(f"{s}:{note}")

        if present.sum() == 0:
            rec.update(status="empty", note=";".join(notes))
            return rec, meta_rows

        tmp = dst.with_suffix(".tmp.npz")
        save = np.savez_compressed if W["COMPRESS"] else np.savez
        save(tmp, vol=arr, present=present, side=np.array(side), slot_uids=np.array([u or "" for u in picks]),
             slot_fallback=np.array(fallback, dtype=np.uint8))
        os.replace(tmp, dst)
        rec.update(slots="".join(str(int(x)) for x in present), side=side, side_src=side_src,
                   fallback="".join(str(int(x)) for x in fallback), note=";".join(notes))

        return rec, meta_rows

    except Exception:
        rec.update(status="fail", note=traceback.format_exc(limit=2).replace("\n", " | "))
        return rec, meta_rows

## Run

In [ ]:
studies = sorted(STUDY_ROWS)

if CFG["N_SHARDS"] > 1:
    studies = [s for s in studies if zlib.crc32(s.encode()) % CFG["N_SHARDS"] == CFG["SHARD_INDEX"]]
    print(f"shard {CFG['SHARD_INDEX']}/{CFG['N_SHARDS']}: {len(studies)} studies")

if CFG["DEBUG"]:
    studies = studies[: CFG["DEBUG_N"]]
    print(f"DEBUG: {len(studies)} studies only")

todo = [s for s in studies if not (out_path(s).is_file() and out_path(s).stat().st_size > 0)]
print(f"{len(studies)} in scope | {len(studies) - len(todo)} cached | {len(todo)} to build")

In [ ]:
n_workers = CFG["N_WORKERS"] or os.cpu_count() or 2
budget = CFG["TIME_BUDGET_H"] * 3600
t0 = time.time()
records, meta_all, counts = [], [], {}

if todo:
    ctx = mp.get_context("fork")

    with ctx.Pool(n_workers) as pool:
        it = pool.imap_unordered(process_study, todo, chunksize=1)
        bar = tqdm(it, total=len(todo), desc="studies", smoothing=0.02)

        for i, (rec, mrows) in enumerate(bar, 1):
            records.append(rec)
            meta_all.extend(mrows)
            counts[rec["status"]] = counts.get(rec["status"], 0) + 1
            elapsed = time.time() - t0
            rate = elapsed / i
            bar.set_postfix(ok=counts.get("ok", 0), bad=len(records) - counts.get("ok", 0),
                            eta_h=f"{rate * (len(todo) - i) / 3600:.2f}")

            if i in (25, 100) or i % 500 == 0:
                print(f"[{elapsed/60:6.1f} min] {i}/{len(todo)} | {rate:.2f} s/study | "
                      f"projected total {rate * len(todo) / 3600:.2f} h | {counts}")

            if elapsed > budget:
                print(f"\ntime budget reached at {i}/{len(todo)}; rerun to resume.")
                pool.terminate()
                break

print(f"\ndone in {(time.time() - t0)/60:.1f} min | {counts}")

## Model

In [ ]:
import torch
import torch.nn as nn
import timm

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]

MODEL_CFG = {
    "BACKBONE": "convnext_tiny",
    "N_SLOTS": 6,
    "ATTN_DIM": 128,
    "DROPOUT": 0.2,
    "SLICE_STRIDE": 2,
    "IMG": 320,
}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device", DEVICE)


class AttnPool(nn.Module):
    """Gated attention over slices. One set of weights, shared by all slots."""

    def __init__(self, dim, hidden):
        super().__init__()
        self.v = nn.Linear(dim, hidden)
        self.u = nn.Linear(dim, hidden)
        self.w = nn.Linear(hidden, 1)

    def forward(self, x):                       # (B, S, D)
        a = self.w(torch.tanh(self.v(x)) * torch.sigmoid(self.u(x)))
        a = a.softmax(dim=1)
        return (a * x).sum(dim=1), a.squeeze(-1)


class Model2p5D(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n_slots = cfg["N_SLOTS"]
        self.backbone = timm.create_model(cfg["BACKBONE"], pretrained=False, num_classes=0, in_chans=3)
        dim = self.backbone.num_features
        self.pool = AttnPool(dim, cfg["ATTN_DIM"])
        self.drop = nn.Dropout(cfg["DROPOUT"])
        self.head = nn.Linear(dim * self.n_slots, len(TARGETS))

    def forward(self, x, present=None):
        b, k, s = x.shape[:3]
        f = self.backbone(x.reshape(b * k * s, *x.shape[3:]))
        f = f.reshape(b * k, s, -1)
        emb, _ = self.pool(f)
        emb = emb.reshape(b, k, -1)
        if present is not None:
            emb = emb * present.unsqueeze(-1)
        return self.head(self.drop(emb.reshape(b, -1)))

## Checkpoints

One directory for all fold.

In [ ]:
N_FOLDS = 5


def find_checkpoints():
    """Finds five fold checkpoints of one training run.

    Searches for `best_fold*.pt` and groups the hits by directory. 
    Only a directory holding all N_FOLDS folds is usable.

    Returns:
        The fold checkpoint paths, ordered by fold number.

    Raises:
        SystemExit: No directory holds a complete set of folds.
    """

    by_dir = {}

    for base in [Path("/kaggle/input"), Path("weights"), Path("runs"), Path(".")]:
        if base.is_dir():
            for path in sorted(base.rglob("best_fold*.pt")):
                by_dir.setdefault(path.parent, {})[path.stem.replace("best_fold", "")] = path

    complete = {d: v for d, v in by_dir.items() if len(v) == N_FOLDS}

    if len(complete) != 1:
        raise SystemExit(
            f"need exactly one directory with all {N_FOLDS} folds, found {len(complete)}: "
            f"{ {str(d): sorted(v) for d, v in by_dir.items()} }"
        )

    folds = next(iter(complete.values()))

    return [folds[k] for k in sorted(folds)]


def load_models(paths):
    """Build and loads one model per checkpoint.

    Args:
        paths: Fold checkpoint paths.

    Returns:
        The loaded models.
    """

    models = []

    for path in paths:
        model = Model2p5D(MODEL_CFG).to(DEVICE).eval()
        state = torch.load(path, map_location="cpu")
        state = state.get("model", state.get("state_dict", state))
        model.load_state_dict(state, strict=True)
        models.append(model)
    
    return models


CKPTS = find_checkpoints()
MODELS = load_models(CKPTS)

print(f"{len(MODELS)} models loaded from {CKPTS[0].parent}")

## Fixing Slice Direction

Some slot may be stored back-to-front relative to their same-plane reference. The is caused by upstream preprocessing.

In [ ]:
REF_SLOT = {3: 0, 5: 0, 4: 1}
MIN_MARGIN = 0.02


def _descriptors(vol):
    """Reduces a slot to one coarse unit-norm vector per slice. 
    
    Dot product becomes a 'correlation'.

    Args:
        vol: One slot's slices, shape (S, H, W).

    Returns:
        Mean-removed, unit-norm descriptors of shape (S, 400).
    """

    s, h, w = vol.shape
    x = vol.reshape(s, 20, h // 20, 20, w // 20).mean(axis=(2, 4)).reshape(s, -1).astype(np.float64)
    x -= x.mean(axis=1, keepdims=True)
    n = np.linalg.norm(x, axis=1, keepdims=True)

    return x / np.where(n > 0, n, 1.0)


def slice_flips(vol, present):
    """Finds reversed slot studies.

    Args:
        vol: The study's cache array, shape (slots, S, H, W).
        present: Per-slot presence flags.

    Returns:
        Slot indices whose slice order should be reversed.
    """

    flips, desc = [], {}
    
    for slot, ref in REF_SLOT.items():

        if not (present[slot] and present[ref]):
            continue

        for k in (slot, ref):
            if k not in desc:
                desc[k] = _descriptors(vol[k])

        forward = float((desc[ref] * desc[slot]).sum(axis=1).mean())
        reverse = float((desc[ref] * desc[slot][::-1]).sum(axis=1).mean())

        if reverse - forward > MIN_MARGIN:
            flips.append(slot)

    return flips

## Predict

In [ ]:
from scipy.stats import rankdata

TTA_OFFSETS = [0, 1]
RESERVE_S = 900


def load_study(study, offsets):
    """Loads one cached study as a batch of TTA views.

    Args:
        study: StudyInstanceUID.
        offsets: Slice-sampling offsets, one view each.

    Returns:
        A (views, present) pair, where views has shape (len(offsets), slots, S, 3, H, W), or
        None if the study has no cache file.
    """

    path = Path(CFG["OUT_DIR"]) / f"{study}.npz"

    if not path.is_file():
        return None

    with np.load(path) as d:
        vol, present = d["vol"], d["present"].astype(np.float32)

    vol = vol.copy()

    for slot in slice_flips(vol, present):
        vol[slot] = vol[slot][::-1]

    n = vol.shape[1]
    base = np.arange(0, n, MODEL_CFG["SLICE_STRIDE"])
    views = []

    for off in offsets:
        idx = np.clip(base + off, 0, n - 1)
        x = np.stack([vol[:, np.clip(idx - 1, 0, n - 1)],
                      vol[:, idx],
                      vol[:, np.clip(idx + 1, 0, n - 1)]], axis=2).astype(np.float32) / 255.0
        views.append((x - 0.449) / 0.226)

    return np.stack(views), present


@torch.no_grad()
def predict(models, studies, offsets):
    """Runs the ensemble over a list of studies, averaging each model over its TTA views.

    Args:
        models: The loaded fold models.
        studies: StudyInstanceUIDs, in submission order.
        offsets: Slice-sampling offsets to average over.

    Returns:
        Probabilities of shape (len(models), len(studies), 12). Rows for studies with no cache
        file are NaN.
    """

    out = np.full((len(models), len(studies), len(TARGETS)), np.nan, np.float32)
    t0 = time.time()

    for i, study in enumerate(studies):
        loaded = load_study(study, offsets)

        if loaded is None:
            continue
        x, present = loaded
        xb = torch.from_numpy(x).to(DEVICE)
        pb = torch.from_numpy(present).to(DEVICE).unsqueeze(0).expand(len(offsets), -1)

        for m, model in enumerate(models):
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                logits = model(xb, pb)
            out[m, i] = torch.sigmoid(logits.float()).mean(dim=0).cpu().numpy()

        if (i + 1) % 100 == 0:
            rate = (time.time() - t0) / (i + 1)
            print(f"  {i+1}/{len(studies)} | {rate:.2f} s/study | "
                  f"eta {rate * (len(studies) - i - 1) / 60:.1f} min", flush=True)

    return out


def rank_mean(preds):
    """Averages several models' predictions by per-label rank.

    Args:
        preds: Probabilities of shape (models, studies, 12), finite.

    Returns:
        Mean ranks in (0, 1), shape (studies, 12).
    """

    n_studies = preds.shape[1]

    return rankdata(preds, axis=1).mean(axis=0) / (n_studies + 1.0)


def fit_budget(unit_cost, n_studies, seconds_left, n_models, n_offsets):
    """Ensure under time budget.

    Args:
        unit_cost: Seconds per study, per model, per TTA pass.
        n_studies: Studies still to predict.
        seconds_left: Wall clock remaining, minus the reserve.
        n_models: Models available.
        n_offsets: TTA passes wanted.

    Returns:
        The (models, offsets) counts to actually run, each at least 1.
    """

    def cost(models, offsets):
        return unit_cost * models * offsets * n_studies

    offsets = n_offsets

    while offsets > 1 and cost(n_models, offsets) > seconds_left:
        offsets -= 1

    models = n_models

    while models > 1 and cost(models, offsets) > seconds_left:
        models -= 1

    return models, offsets

## Write `submission.csv`

In [ ]:
sub_path = ROOT / "sample_submission.csv"
sub = pd.read_csv(sub_path) if sub_path.is_file() else pd.DataFrame({"StudyInstanceUID": sorted(STUDY_ROWS)})
studies = sub["StudyInstanceUID"].astype(str).tolist()
print(f"{len(studies)} studies to predict")

probe = studies[:min(10, len(studies))]
t0 = time.time()
predict(MODELS, probe, TTA_OFFSETS)
unit_cost = (time.time() - t0) / (len(probe) * len(MODELS) * len(TTA_OFFSETS))

left = 9 * 3600 - (time.time() - NB_T0) - RESERVE_S
n_models, n_offsets = fit_budget(unit_cost, len(studies), left, len(MODELS), len(TTA_OFFSETS))
print(f"probe: {unit_cost * len(MODELS) * len(TTA_OFFSETS):.2f} s/study | {left/3600:.2f} h left")
if (n_models, n_offsets) != (len(MODELS), len(TTA_OFFSETS)):
    print(f"OVER BUDGET: trimmed to {n_models} models x {n_offsets} TTA")
print(f"running {n_models} models x {n_offsets} TTA | "
      f"projected {unit_cost * n_models * n_offsets * len(studies) / 3600:.2f} h")

In [ ]:
preds = predict(MODELS[:n_models], studies, TTA_OFFSETS[:n_offsets])

ok = np.isfinite(preds).all(axis=(0, 2))
print(f"{ok.sum()}/{len(studies)} studies predicted; {(~ok).sum()} fall back to 0.5")

final = np.full((len(studies), len(TARGETS)), 0.5, np.float32)
if ok.any():
    final[ok] = rank_mean(preds[:, ok])

out = pd.DataFrame(final, columns=TARGETS)
out.insert(0, "StudyInstanceUID", studies)
out.to_csv("/kaggle/working/submission.csv", index=False)
print(out.shape)
out.head()

In [ ]:
# A constant column scores 0.5 AUC, and a code competition gives no second chance.

chk = pd.read_csv("/kaggle/working/submission.csv")

nunique = chk[TARGETS].nunique()

print("distinct values per label:\n", nunique.to_string())

if len(chk) > 5:
    assert (nunique > 1).all(), f"constant column(s): {nunique[nunique <= 1].index.tolist()}"

print(f"\nOK | {len(chk)} rows | total kernel time {(time.time() - NB_T0)/3600:.2f} h")
